In [ ]:
pip install requests pandas numpy pandas-datareader scipy arch yfinance

In [1]:
import requests
import pandas as pd
import numpy as np
from IPython.display import display
import pandas_datareader.data as pdr
from scipy.stats import norm
from arch import arch_model
import yfinance as yf

SYMBOLS = ["HDFCBANK", "NESTLEIND"]

#### NSE Option Chain

Two API calls are made per stock:  
- **Call A:** Fetches contract info to get all available expiry dates; we keep only the first two.  
- **Call B:** Fetches the full option chain for each of those two expiries.

Spot Price Extraction
The spot price is pulled directly from the underlyingValue field in the NSE JSON response.

For every option at every strike, the fair price is determined by priority:  
1. **Primary:** Last Traded Price (LTP) - if > 0  
2. **Secondary:** Mid-price - if Bid & Ask both exist  
3. **Discard:** NaN - strike is illiquid and excluded

Moneyness = Strike / Spot is computed for every row, then split into:  
- **ATM:** Moneyness between 0.98 and 1.02 (within ±2% of spot)  
- **OTM:** Calls with strike above spot; Puts with strike below spot

- **Step 1:** Look for strikes 5–10% OTM.  
- **Step 2:** If that range is empty, step in and look at 2–5% OTM instead.

In [2]:
# ---------------- NSE CONFIG ----------------
_NSE_OC_PAGE  = "https://www.nseindia.com/option-chain"
_NSE_CONTRACT = "https://www.nseindia.com/api/option-chain-contract-info?symbol={symbol}"
_NSE_OC_V3    = "https://www.nseindia.com/api/option-chain-v3?type=Equity&symbol={symbol}&expiry={expiry}"

_NSE_HEADERS = {
    "user-agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36"
    ),
    "accept-language": "en,gu;q=0.9,hi;q=0.8",
    "accept-encoding": "gzip, deflate, br",
    "referer": "https://www.nseindia.com/option-chain",
}

def _nse_session():
    """Warms up a session by visiting the home page and option chain page."""
    s = requests.Session()
    s.get("https://www.nseindia.com", headers=_NSE_HEADERS, timeout=10)
    s.get(_NSE_OC_PAGE, headers=_NSE_HEADERS, timeout=10)
    return s, dict(s.cookies)

def _nse_get(session, cookies, url):
    """GET with automatic session refresh on 401/403."""
    resp = session.get(url, headers=_NSE_HEADERS, timeout=10, cookies=cookies)
    if resp.status_code in [401, 403]:
        session.close()
        session, cookies = _nse_session()
        resp = session.get(url, headers=_NSE_HEADERS, timeout=10, cookies=cookies)
        print("    [NSE] Session refreshed...")
    return session, cookies, resp

def get_option_chain_and_spot(symbol: str):
    """Fetches the option chain and extracts the live spot price from NSE."""
    session, cookies = _nse_session()
    url_exp = _NSE_CONTRACT.format(symbol=symbol)
    session, cookies, resp = _nse_get(session, cookies, url_exp)
    info = resp.json()
    expiry_dates = (
        info.get("expiryDates") or info.get("records", {}).get("expiryDates", [])
    )[:2]

    all_rows = []
    underlying_spot = None

    for expiry in expiry_dates:
        url_oc = _NSE_OC_V3.format(symbol=symbol, expiry=expiry)
        session, cookies, resp = _nse_get(session, cookies, url_oc)
        json_data = resp.json()

        if underlying_spot is None:
            underlying_spot = json_data.get("records", {}).get("underlyingValue")

        records = json_data.get("records", {}).get("data", [])
        for entry in records:
            strike = float(entry.get("strikePrice", 0))
            for opt_type in ("CE", "PE"):
                d = entry.get(opt_type)
                if not d:
                    continue
                ltp = float(d.get("lastPrice", 0) or 0)
                bid = float(d.get("bidprice", 0) or 0)
                ask = float(d.get("askPrice", 0) or 0)
                price = ltp if ltp > 0 else (
                    (bid + ask) / 2 if (bid > 0 and ask > 0) else np.nan
                )
                all_rows.append({
                    "expiry": pd.Timestamp(expiry),
                    "strike": strike,
                    "type":   opt_type,
                    "price":  price,
                    "IV":     float(d.get("impliedVolatility", np.nan) or np.nan),
                })

    session.close()
    return pd.DataFrame(all_rows), underlying_spot

#### Fetch Option Chains and Build Results Table

For each stock, the two nearest expiries are fetched. Strikes are classified as ATM (±2%) or OTM (5–10%, falling back to 2–5%) for both Calls and Puts.

In [3]:
today = pd.Timestamp.today().normalize()
rows  = []

for symbol in SYMBOLS:
    print(f"Fetching {symbol}...")
    try:
        chain_df, spot = get_option_chain_and_spot(symbol)
    except Exception as e:
        print(f"  Error: {e}")
        continue

    if chain_df.empty or spot is None:
        print(f"  No data returned.")
        continue

    print(f"  Spot: Rs. {spot:,.2f}")
    chain_df["moneyness"] = chain_df["strike"] / spot

    for expiry in sorted(chain_df["expiry"].unique()):
        exp_ts      = pd.Timestamp(expiry)
        days_to_exp = (exp_ts - today).days
        exp_df      = chain_df[chain_df["expiry"] == exp_ts]

        for opt_type in ["CE", "PE"]:
            type_label = "Call" if opt_type == "CE" else "Put"

            atm = exp_df[
                (exp_df["type"] == opt_type) &
                (exp_df["moneyness"].between(0.98, 1.02)) &
                (exp_df["price"].notna())
            ]

            far_range = (1.05, 1.10) if opt_type == "CE" else (0.90, 0.95)
            otm = exp_df[
                (exp_df["type"] == opt_type) &
                (exp_df["moneyness"].between(*far_range)) &
                (exp_df["price"].notna())
            ]
            otm_label = "OTM (5-10%)"
            if otm.empty:
                near_range = (1.02, 1.05) if opt_type == "CE" else (0.95, 0.98)
                otm = exp_df[
                    (exp_df["type"] == opt_type) &
                    (exp_df["moneyness"].between(*near_range)) &
                    (exp_df["price"].notna())
                ]
                otm_label = "OTM (2-5%)"

            for cat_label, cat_df in [("ATM (±2%)", atm), (otm_label, otm)]:
                for _, row in cat_df.iterrows():
                    rows.append({
                        "Stock":              symbol,
                        "Expiry":             exp_ts.strftime("%d-%b-%Y"),
                        "Days to Expiry":     days_to_exp,
                        "Call / Put":         type_label,
                        "ATM / OTM":          cat_label,
                        "Strike":             row["strike"],
                        "Market Price (Rs.)": round(row["price"], 2),
                        "Impl. Vol (%)": round(row["IV"], 2) if not pd.isna(row["IV"]) else None,
                        "Spot Price (Rs.)":   spot,
                    })

print("\nDone.")

Fetching HDFCBANK...
  Spot: Rs. 810.00
Fetching NESTLEIND...
  Spot: Rs. 1,256.90

Done.


#### Full Option Chain Results

One table per stock per expiry. **Impl. Vol (%)** is the implied volatility reported directly by NSE.

In [4]:
result_df = pd.DataFrame(rows)

for symbol in SYMBOLS:
    sym_df = result_df[result_df["Stock"] == symbol]
    if sym_df.empty:
        print(f"No data for {symbol}.")
        continue

    for expiry in sym_df["Expiry"].unique():
        exp_df = sym_df[sym_df["Expiry"] == expiry]
        days   = exp_df["Days to Expiry"].iloc[0]
        spot   = exp_df["Spot Price (Rs.)"].iloc[0]
        display(
            exp_df[
                ["Call / Put", "ATM / OTM", "Strike",
                 "Market Price (Rs.)", "Impl. Vol (%)", "Spot Price (Rs.)", "Days to Expiry"]
            ]
            .reset_index(drop=True)
            .style
            .set_caption(
                f"{symbol}  |  Expiry: {expiry}  |  Days to Expiry: {days}  |  Spot: Rs. {spot:,.2f}"
            )
            .hide(axis="index")
            .format({
                "Strike":             "{:,.2f}",
                "Market Price (Rs.)": "{:.2f}",
                "Spot Price (Rs.)":   "{:,.2f}",
                "Impl. Vol (%)": lambda x: f"{x:.2f}" if x is not None else "N/A",
            })
        )

Call / Put,ATM / OTM,Strike,Market Price (Rs.),Impl. Vol (%),Spot Price (Rs.),Days to Expiry
Call,ATM (±2%),795.00,27.75,28.86,810.00,13
Call,ATM (±2%),800.00,24.55,28.73,810.00,13
Call,ATM (±2%),805.00,21.70,28.82,810.00,13
Call,ATM (±2%),810.00,19.00,28.79,810.00,13
Call,ATM (±2%),815.00,16.70,29.04,810.00,13
Call,ATM (±2%),820.00,14.40,28.93,810.00,13
Call,ATM (±2%),825.00,12.60,29.29,810.00,13
Call,OTM (5-10%),855.00,5.30,31.19,810.00,13
Call,OTM (5-10%),860.00,4.55,31.49,810.00,13
Call,OTM (5-10%),865.00,3.85,31.66,810.00,13


Call / Put,ATM / OTM,Strike,Market Price (Rs.),Impl. Vol (%),Spot Price (Rs.),Days to Expiry
Call,ATM (±2%),795.00,39.70,24.41,810.00,41
Call,ATM (±2%),800.00,37.20,24.92,810.00,41
Call,ATM (±2%),805.00,36.30,nan,810.00,41
Call,ATM (±2%),810.00,31.70,25.02,810.00,41
Call,ATM (±2%),815.00,29.60,25.46,810.00,41
Call,ATM (±2%),820.00,26.95,25.26,810.00,41
Call,ATM (±2%),825.00,26.00,26.51,810.00,41
Call,OTM (5-10%),855.00,15.50,27.07,810.00,41
Call,OTM (5-10%),860.00,13.40,26.38,810.00,41
Call,OTM (5-10%),865.00,12.50,26.83,810.00,41


Call / Put,ATM / OTM,Strike,Market Price (Rs.),Impl. Vol (%),Spot Price (Rs.),Days to Expiry
Call,ATM (±2%),"1,240.00",36.90,26.44,"1,256.90",13
Call,ATM (±2%),"1,250.00",30.95,26.40,"1,256.90",13
Call,ATM (±2%),"1,260.00",26.20,26.97,"1,256.90",13
Call,ATM (±2%),"1,270.00",20.60,25.97,"1,256.90",13
Call,ATM (±2%),"1,280.00",16.95,26.38,"1,256.90",13
Call,OTM (5-10%),"1,320.00",6.35,26.48,"1,256.90",13
Call,OTM (5-10%),"1,330.00",4.30,nan,"1,256.90",13
Call,OTM (5-10%),"1,350.00",2.75,26.84,"1,256.90",13
Call,OTM (5-10%),"1,370.00",1.40,26.71,"1,256.90",13
Call,OTM (5-10%),"1,380.00",0.90,26.27,"1,256.90",13


Call / Put,ATM / OTM,Strike,Market Price (Rs.),Impl. Vol (%),Spot Price (Rs.),Days to Expiry
Call,ATM (±2%),"1,270.00",35.00,20.51,"1,256.90",41
Call,OTM (5-10%),"1,320.00",20.00,23.11,"1,256.90",41
Call,OTM (5-10%),"1,350.00",13.05,23.51,"1,256.90",41
Call,OTM (5-10%),"1,360.00",10.20,22.75,"1,256.90",41
Put,ATM (±2%),"1,240.00",30.00,26.31,"1,256.90",41
Put,OTM (5-10%),"1,170.00",18.70,nan,"1,256.90",41


#### Option Selection for BSM Analysis

From **both expiries** of each stock, four options are selected per expiry:

| Slot | Rule |
|---|---|
| ATM Call | Strike with smallest \|Strike − Spot\| among ATM calls |
| ATM Put  | Strike with smallest \|Strike − Spot\| among ATM puts  |
| OTM Call | Middle strike of the OTM call range |
| OTM Put  | Middle strike of the OTM put range  |


This gives **4 options × 2 expiries × 2 stocks = 16 rows** for BSM analysis.

In [ ]:
selected_rows = []

for symbol in SYMBOLS:
    stock_rows = [r for r in rows if r["Stock"] == symbol]
    if not stock_rows:
        continue

    df_s = pd.DataFrame(stock_rows)
    sorted_expiries = sorted(df_s["Expiry"].unique(), key=lambda x: pd.Timestamp(x))

    for expiry in sorted_expiries:
        df_exp = df_s[df_s["Expiry"] == expiry].copy()
        spot   = df_exp["Spot Price (Rs.)"].iloc[0]

        for opt_type in ["Call", "Put"]:
            # ATM: single strike closest to spot
            atm_df = df_exp[
                (df_exp["Call / Put"] == opt_type) &
                (df_exp["ATM / OTM"].str.startswith("ATM"))
            ].copy()
            if not atm_df.empty:
                atm_df["_dist"] = (atm_df["Strike"] - spot).abs()
                best = atm_df.nsmallest(1, "_dist").drop(columns="_dist")
                selected_rows.append(best.iloc[0].to_dict())

            # OTM: middle strike of the OTM range
            otm_df = df_exp[
                (df_exp["Call / Put"] == opt_type) &
                (df_exp["ATM / OTM"].str.startswith("OTM"))
            ].reset_index(drop=True)
            if not otm_df.empty:
                mid_idx = len(otm_df) // 2
                selected_rows.append(otm_df.iloc[mid_idx].to_dict())

df_selected = pd.DataFrame(selected_rows)
print(f"Selected {len(selected_rows)} options for BSM analysis:")
display(
    df_selected[
        ["Stock", "Expiry", "Days to Expiry", "Call / Put", "ATM / OTM",
         "Strike", "Market Price (Rs.)", "Impl. Vol (%)", "Spot Price (Rs.)"]
    ]
    .reset_index(drop=True)
    .style
    .set_caption("Selected Options — 1 ATM Call, 1 ATM Put, 1 OTM Call, 1 OTM Put  |  per stock × per expiry  |  16 total")
    .hide(axis="index")
    .format({
        "Strike":             "{:,.2f}",
        "Market Price (Rs.)": "{:.2f}",
        "Spot Price (Rs.)":   "{:,.2f}",
        "Impl. Vol (%)": lambda x: f"{x:.2f}" if x is not None else "N/A",
    })
)

Selected 16 options for BSM analysis:


Stock,Expiry,Days to Expiry,Call / Put,ATM / OTM,Strike,Market Price (Rs.),Impl. Vol (%),Spot Price (Rs.)
HDFCBANK,28-Apr-2026,13,Call,ATM (±2%),810.00,19.00,28.79,810.00
HDFCBANK,28-Apr-2026,13,Call,OTM (5-10%),875.00,3.00,32.86,810.00
HDFCBANK,28-Apr-2026,13,Put,ATM (±2%),810.00,19.00,33.53,810.00
HDFCBANK,28-Apr-2026,13,Put,OTM (5-10%),750.00,3.70,38.14,810.00
HDFCBANK,26-May-2026,41,Call,ATM (±2%),810.00,31.70,25.02,810.00
HDFCBANK,26-May-2026,41,Call,OTM (5-10%),875.00,10.05,26.71,810.00
HDFCBANK,26-May-2026,41,Put,ATM (±2%),810.00,29.60,31.52,810.00
HDFCBANK,26-May-2026,41,Put,OTM (5-10%),745.00,8.60,32.07,810.00
NESTLEIND,28-Apr-2026,13,Call,ATM (±2%),"1,260.00",26.20,26.97,"1,256.90"
NESTLEIND,28-Apr-2026,13,Call,OTM (5-10%),"1,350.00",2.75,26.84,"1,256.90"


In [ ]:
VOL_WINDOW = 20
LOOKBACK_MONTHS = 6
FALLBACK_RFR = 0.07

def get_india_rfr(fallback=FALLBACK_RFR):
    """Fetches India 10Y Yield from FRED using pandas-datareader."""
    try:
        end = pd.Timestamp.today()
        start = end - pd.DateOffset(months=3)
        df = pdr.DataReader("INDIRLTLT01STM", "fred", start, end)
        if df.empty: return fallback
        return float(df.dropna().iloc[-1].iloc[0]) / 100
    except:
        return fallback

def get_stock_volatilities(yf_symbol):
    """Calculates Historical Vol (Part A Mean) and GARCH(1,1) Vol."""
    end = pd.Timestamp.today()
    start = end - pd.DateOffset(months=LOOKBACK_MONTHS)

    data = yf.download(yf_symbol, start=start.strftime('%Y-%m-%d'),
                        end=end.strftime('%Y-%m-%d'), interval='1d', progress=False)
    if data.empty:
        return np.nan, np.nan

    close = data['Close'].squeeze()
    log_returns = np.log(close / close.shift(1)).dropna()

    rolling_vol = log_returns.rolling(window=VOL_WINDOW).std() * np.sqrt(252)
    hist_vol = float(rolling_vol.mean())

    garch_input = log_returns * 100
    try:
        model = arch_model(garch_input, vol='Garch', p=1, q=1, dist='t')
        res = model.fit(disp='off')
        garch_vol = (float(res.conditional_volatility.iloc[-1]) / 100) * np.sqrt(252)
    except:
        garch_vol = np.nan

    return hist_vol, garch_vol

In [7]:
def bsm_pricing(S, K, T, r, sigma, option_type='Call'):
    """Calculates BSM price and intermediate variables."""
    if pd.isna(sigma) or T <= 0 or sigma <= 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type.lower() == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return price, d1, d2, norm.cdf(d1), norm.cdf(d2)

#### BSM Pricing on Selected Options

Volatilities are fetched once per stock. BSM is computed twice - using Historical Vol and (corrected) GARCH Vol - for each of the 16 selected options (4 per expiry × 2 expiries × 2 stocks).

In [8]:
YF_MAP = {"HDFCBANK": "HDFCBANK.NS", "NESTLEIND": "NESTLEIND.NS"}
rfr = get_india_rfr()

intermediate_data = []
comparison_data   = []

for symbol in SYMBOLS:
    h_vol, g_vol = get_stock_volatilities(YF_MAP[symbol])
    stock_rows = [r for r in selected_rows if r["Stock"] == symbol]

    for row in stock_rows:
        S     = float(row["Spot Price (Rs.)"])
        K     = float(row["Strike"])
        T     = float(row["Days to Expiry"]) / 365
        otype = row["Call / Put"]

        p_hist,  d1, d2, nd1, nd2 = bsm_pricing(S, K, T, rfr, h_vol, otype)
        p_garch, _,  _,  _,   _  = bsm_pricing(S, K, T, rfr, g_vol, otype)

        intermediate_data.append({
            "Stock": symbol, "Expiry": row["Expiry"], "Type": otype,
            "Strike": K, "Spot": S, "Vol (Hist)": h_vol,
            "T (Yrs)": T, "d1": d1, "d2": d2, "N(d1)": nd1, "N(d2)": nd2
        })

        comparison_data.append({
            "Stock": symbol, "Expiry": row["Expiry"], "Type": otype, "Strike": K,
            "Market Price": row["Market Price (Rs.)"],
            "BSM (Hist Vol)": p_hist, "BSM (GARCH Vol)": p_garch
        })

df_inter   = pd.DataFrame(intermediate_data)
df_compare = pd.DataFrame(comparison_data)

C:\Users\Jairam Ayyar\AppData\Local\Temp\ipykernel_16928\1325372373.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(yf_symbol, start=start.strftime('%Y-%m-%d'),
C:\Users\Jairam Ayyar\AppData\Local\Temp\ipykernel_16928\1325372373.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(yf_symbol, start=start.strftime('%Y-%m-%d'),


In [9]:
print("### Intermediate BSM Calculations (Using Historical Vol)")
display(df_inter.style.format({
    "Strike": "{:,.2f}", "Spot": "{:,.2f}", "Vol (Hist)": "{:.2%}",
    "T (Yrs)": "{:.4f}", "d1": "{:.4f}", "d2": "{:.4f}",
    "N(d1)": "{:.4f}", "N(d2)": "{:.4f}"
}).hide(axis='index'))

### Intermediate BSM Calculations (Using Historical Vol)


Stock,Expiry,Type,Strike,Spot,Vol (Hist),T (Yrs),d1,d2,N(d1),N(d2)
HDFCBANK,28-Apr-2026,Call,810.00,810.00,17.30%,0.0356,0.0903,0.0577,0.5360,0.5230
HDFCBANK,28-Apr-2026,Call,875.00,810.00,17.30%,0.0356,-2.2735,-2.3061,0.0115,0.0106
HDFCBANK,28-Apr-2026,Put,810.00,810.00,17.30%,0.0356,0.0903,0.0577,0.5360,0.5230
HDFCBANK,28-Apr-2026,Put,750.00,810.00,17.30%,0.0356,2.4471,2.4144,0.9928,0.9921
HDFCBANK,26-May-2026,Call,810.00,810.00,17.30%,0.1123,0.1604,0.1024,0.5637,0.5408
HDFCBANK,26-May-2026,Call,875.00,810.00,17.30%,0.1123,-1.1706,-1.2286,0.1209,0.1096
HDFCBANK,26-May-2026,Put,810.00,810.00,17.30%,0.1123,0.1604,0.1024,0.5637,0.5408
HDFCBANK,26-May-2026,Put,745.00,810.00,17.30%,0.1123,1.6028,1.5448,0.9455,0.9388
NESTLEIND,28-Apr-2026,Call,"1,260.00","1,256.90",16.00%,0.0356,0.0135,-0.0167,0.5054,0.4934
NESTLEIND,28-Apr-2026,Call,"1,350.00","1,256.90",16.00%,0.0356,-2.2709,-2.3011,0.0116,0.0107


In [10]:
print("### Final Option Price Comparison")
display(df_compare.style.format({
    "Strike": "{:,.2f}", "Market Price": "{:.2f}",
    "BSM (Hist Vol)": "{:.2f}", "BSM (GARCH Vol)": "{:.2f}"
}).hide(axis='index'))

### Final Option Price Comparison


Stock,Expiry,Type,Strike,Market Price,BSM (Hist Vol),BSM (GARCH Vol)
HDFCBANK,28-Apr-2026,Call,810.00,19.00,11.55,28.47
HDFCBANK,28-Apr-2026,Call,875.00,3.00,0.10,7.48
HDFCBANK,28-Apr-2026,Put,810.00,19.00,9.59,26.51
HDFCBANK,28-Apr-2026,Put,750.00,3.70,0.06,6.27
HDFCBANK,26-May-2026,Call,810.00,31.70,21.90,51.78
HDFCBANK,26-May-2026,Call,875.00,10.05,2.73,26.65
HDFCBANK,26-May-2026,Put,810.00,29.60,15.75,45.63
HDFCBANK,26-May-2026,Put,745.00,8.60,1.11,19.60
NESTLEIND,28-Apr-2026,Call,"1,260.00",26.20,15.11,16.25
NESTLEIND,28-Apr-2026,Call,"1,350.00",2.75,0.15,0.25


In [ ]:
master_records = []

for symbol in SYMBOLS:
    h_vol, g_vol = get_stock_volatilities(YF_MAP[symbol])
    stock_rows = [r for r in selected_rows if r["Stock"] == symbol]

    for row in stock_rows:
        S     = float(row["Spot Price (Rs.)"])
        K     = float(row["Strike"])
        T     = float(row["Days to Expiry"]) / 365
        otype = row["Call / Put"]
        cat   = row["ATM / OTM"]
        mkt_p = float(row["Market Price (Rs.)"])

        p_hist,  d1, d2, nd1, nd2 = bsm_pricing(S, K, T, rfr, h_vol, otype)
        p_garch, _,  _,  _,   _  = bsm_pricing(S, K, T, rfr, g_vol, otype)

        master_records.append({
            "Stock":        symbol,
            "Expiry":       row["Expiry"],
            "Type":         otype,
            "Moneyness":    cat,
            "Spot (S)":     S,
            "Strike (K)":   K,
            "RFR (r)":      rfr,
            "Vol (Hist)":   h_vol,
            "Vol (GARCH)":  g_vol,
            "T (Years)":    T,
            "d1":           d1,
            "d2":           d2,
            "N(d1)":        nd1,
            "N(d2)":        nd2,
            "Market Price": mkt_p,
            "BSM (Hist)":   p_hist,
            "BSM (GARCH)":  p_garch
        })

df_master = pd.DataFrame(master_records)

print(f"### Master BSM Analysis Report (Execution Date: {today.strftime('%d-%b-%Y')})")
display(df_master.style.format({
    "Spot (S)":     "{:,.2f}",
    "Strike (K)":   "{:,.2f}",
    "RFR (r)":      "{:.2%}",
    "Vol (Hist)":   "{:.2%}",
    "Vol (GARCH)":  "{:.2%}",
    "T (Years)":    "{:.4f}",
    "d1":           "{:.4f}",
    "d2":           "{:.4f}",
    "N(d1)":        "{:.4f}",
    "N(d2)":        "{:.4f}",
    "Market Price": "{:.2f}",
    "BSM (Hist)":   "{:.2f}",
    "BSM (GARCH)":  "{:.2f}"
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#f2f2f2'), ('color', 'black'), ('font-weight', 'bold')]}
]).hide(axis='index'))

### Master BSM Analysis Report (Execution Date: 15-Apr-2026)


C:\Users\Jairam Ayyar\AppData\Local\Temp\ipykernel_16928\1325372373.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(yf_symbol, start=start.strftime('%Y-%m-%d'),
C:\Users\Jairam Ayyar\AppData\Local\Temp\ipykernel_16928\1325372373.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(yf_symbol, start=start.strftime('%Y-%m-%d'),


Stock,Expiry,Type,Moneyness,Spot (S),Strike (K),RFR (r),Vol (Hist),Vol (GARCH),T (Years),d1,d2,N(d1),N(d2),Market Price,BSM (Hist),BSM (GARCH)
HDFCBANK,28-Apr-2026,Call,ATM (±2%),810.00,810.00,6.78%,17.30%,45.12%,0.0356,0.0903,0.0577,0.5360,0.5230,19.00,11.55,28.47
HDFCBANK,28-Apr-2026,Call,OTM (5-10%),810.00,875.00,6.78%,17.30%,45.12%,0.0356,-2.2735,-2.3061,0.0115,0.0106,3.00,0.10,7.48
HDFCBANK,28-Apr-2026,Put,ATM (±2%),810.00,810.00,6.78%,17.30%,45.12%,0.0356,0.0903,0.0577,0.5360,0.5230,19.00,9.59,26.51
HDFCBANK,28-Apr-2026,Put,OTM (5-10%),810.00,750.00,6.78%,17.30%,45.12%,0.0356,2.4471,2.4144,0.9928,0.9921,3.70,0.06,6.27
HDFCBANK,26-May-2026,Call,ATM (±2%),810.00,810.00,6.78%,17.30%,45.12%,0.1123,0.1604,0.1024,0.5637,0.5408,31.70,21.90,51.78
HDFCBANK,26-May-2026,Call,OTM (5-10%),810.00,875.00,6.78%,17.30%,45.12%,0.1123,-1.1706,-1.2286,0.1209,0.1096,10.05,2.73,26.65
HDFCBANK,26-May-2026,Put,ATM (±2%),810.00,810.00,6.78%,17.30%,45.12%,0.1123,0.1604,0.1024,0.5637,0.5408,29.60,15.75,45.63
HDFCBANK,26-May-2026,Put,OTM (5-10%),810.00,745.00,6.78%,17.30%,45.12%,0.1123,1.6028,1.5448,0.9455,0.9388,8.60,1.11,19.60
NESTLEIND,28-Apr-2026,Call,ATM (±2%),"1,256.90","1,260.00",6.78%,16.00%,17.20%,0.0356,0.0135,-0.0167,0.5054,0.4934,26.20,15.11,16.25
NESTLEIND,28-Apr-2026,Call,OTM (5-10%),"1,256.90","1,350.00",6.78%,16.00%,17.20%,0.0356,-2.2709,-2.3011,0.0116,0.0107,2.75,0.15,0.25


In [ ]:
output_filename = "BSM_Analysis_Report_v2.csv"
df_master.to_csv(output_filename, index=False)

print(f"Successfully generated: {output_filename}")
print("-" * 50)
print("Columns: Stock, Expiry, Type, Moneyness, Spot(S), Strike(K), RFR(r),")
print("         Vol(Hist), Vol(GARCH), T(Years), d1, d2, N(d1), N(d2),")
print("         Market Price, BSM(Hist), BSM(GARCH)")
print("-" * 50)
print(f"Total rows: {len(df_master)}  (4 options x 2 expiries x {len(SYMBOLS)} stocks)")
print("\nPreview:")
display(df_master)

Successfully generated: BSM_Analysis_Report_v2.csv
--------------------------------------------------
Columns: Stock, Expiry, Type, Moneyness, Spot(S), Strike(K), RFR(r),
         Vol(Hist), Vol(GARCH), T(Years), d1, d2, N(d1), N(d2),
         Market Price, BSM(Hist), BSM(GARCH)
--------------------------------------------------
Total rows: 16  (4 options x 2 expiries x 2 stocks)

Preview:


,Stock,Expiry,Type,Moneyness,Spot (S),Strike (K),RFR (r),Vol (Hist),Vol (GARCH),T (Years),d1,d2,N(d1),N(d2),Market Price,BSM (Hist),BSM (GARCH)
0,HDFCBANK,28-Apr-2026,Call,ATM (±2%),810.0,810.0,0.067833,0.173032,0.451249,0.035616,0.090312,0.057657,0.535980,0.522989,19.00,11.545225,28.466065
1,HDFCBANK,28-Apr-2026,Call,OTM (5-10%),810.0,875.0,0.067833,0.173032,0.451249,0.035616,-2.273466,-2.306121,0.011499,0.010552,3.00,0.103576,7.481824
2,HDFCBANK,28-Apr-2026,Put,ATM (±2%),810.0,810.0,0.067833,0.173032,0.451249,0.035616,0.090312,0.057657,0.535980,0.522989,19.00,9.590642,26.511482
3,HDFCBANK,28-Apr-2026,Put,OTM (5-10%),810.0,750.0,0.067833,0.173032,0.451249,0.035616,2.447090,2.414435,0.992799,0.992120,3.70,0.063003,6.272214
4,HDFCBANK,26-May-2026,Call,ATM (±2%),810.0,810.0,0.067833,0.173032,0.451249,0.112329,0.160386,0.102394,0.563712,0.540778,31.70,21.901241,51.775578
5,HDFCBANK,26-May-2026,Call,OTM (5-10%),810.0,875.0,0.067833,0.173032,0.451249,0.112329,-1.170640,-1.228632,0.120872,0.109605,10.05,2.729904,26.651611
6,HDFCBANK,26-May-2026,Put,ATM (±2%),810.0,810.0,0.067833,0.173032,0.451249,0.112329,0.160386,0.102394,0.563712,0.540778,29.60,15.752791,45.627128
7,HDFCBANK,26-May-2026,Put,OTM (5-10%),810.0,745.0,0.067833,0.173032,0.451249,0.112329,1.602812,1.544820,0.945512,0.938805,8.60,1.108848,19.599339
8,NESTLEIND,28-Apr-2026,Call,ATM (±2%),1256.9,1260.0,0.067833,0.160031,0.171988,0.035616,0.013532,-0.016669,0.505398,0.493350,26.20,15.114000,16.245406
9,NESTLEIND,28-Apr-2026,Call,OTM (5-10%),1256.9,1350.0,0.067833,0.160031,0.171988,0.035616,-2.270884,-2.301086,0.011577,0.010693,2.75,0.149879,0.253382
